In [1]:
import os

OPENAI_API_KEY = "OPENAI_API_KEY"

llm_config = {
    "model": "gpt-4",
    "api_key": OPENAI_API_KEY
}

In [2]:
## Writing Prompt

In [3]:
task = '''
        Write a concise but engaging blogpost about
      Hope AI . Make sure the blogpost is
       within 100 words.
       '''

In [4]:
##Writer Assistant Agent

In [5]:
import autogen

writer = autogen.AssistantAgent(
    name="Writer",
    system_message="You are a writer. You write engaging and concise " 
        "blogpost (with title) on given topics. You must polish your "
        "writing based on the feedback you receive and give a refined "
        "version. Only return your final work without additional comments.",
    llm_config=llm_config,
)

In [6]:
##Direct Draft Generation

In [7]:
reply = writer.generate_reply(messages=[{"content": task,"role": "user"}])
print(reply)

Title: Unleashing the Future: The Dawning of Hope AI

Enter the realm of Hope AI, an emerging technological titan that's reshaping our reality. This breakthrough isn't just about mimicking human intelligence, it's about enhancing it. Surpassing the limitations of traditional AI, Hope AI fuels constructive change by learning from human behaviors, emotions, and intuition. It’s a revolution, enabling smarter decisions and opening fresh possibilities. Optimists muse that it could help resolve complex challenges, from climate change to global health. Witness the dawn of an era where technology and humanity intertwine in an exciting, unprecedented dance - this is the promising horizon of Hope AI.


In [8]:
##Critic Agent

In [9]:
critic = autogen.AssistantAgent(
    name="Critic",
    is_termination_msg=lambda x: x.get("content", "").find("TERMINATE") >= 0,
    llm_config=llm_config,
    system_message="You are a critic. You review the work of "
                "the writer and provide constructive "
                "feedback to help improve the quality of the content.",
)

In [10]:
##Simple Reflection (Two-Agent Chat)

In [11]:
res = critic.initiate_chat(
    recipient=writer,
    message=task,
    max_turns=2,
    summary_method="last_msg"
)

Critic (to Writer):


        Write a concise but engaging blogpost about
      Hope AI . Make sure the blogpost is
       within 100 words.
       

--------------------------------------------------------------------------------
Writer (to Critic):

Title: Unleashing Boundless Potential: Hope in AI 

Artificial Intelligence (AI) is revolutionizing our world, bringing hope of unimaginable progress. AI has the potential to reshape our lives by automating tasks, improving efficiency, and unlocking new possibilities. No sector remains untouched, from medicine to education to transportation. While the AI road has its challenges - ethical issues, potential job loss, etc., the promise of a transformed future brimming with enhanced capabilities and solutions gives us hope. Thus, rather than fear AI, let's embrace this technology, bustling with infinite potential and expand our horizons. The dawn of AI is not a time for fear, but a beacon of hope.

--------------------------------------------

In [12]:
##Specialized Nested Reviewer Agents

In [13]:
SEO_reviewer = autogen.AssistantAgent(
    name="SEO_Reviewer",
    llm_config=llm_config,
    system_message="You are an SEO reviewer... Make sure your suggestion is concise (within 3 bullet points)...",
)

legal_reviewer = autogen.AssistantAgent(
    name="Legal_Reviewer",
    llm_config=llm_config,
    system_message="You are a legal reviewer... ensure that content is legally compliant...",
)

ethics_reviewer = autogen.AssistantAgent(
    name="Ethics_Reviewer",
    llm_config=llm_config,
    system_message="You are an ethics reviewer... ensure that content is ethically sound...",
)

meta_reviewer = autogen.AssistantAgent(
    name="Meta_Reviewer",
    llm_config=llm_config,
    system_message="You are a meta reviewer, you aggragate and review the work of other reviewers and give a final suggestion...",
)

In [14]:
##Structure the Nested Chat Configuration

In [15]:
def reflection_message(recipient, messages, sender, config):
    return f'''Review the following content. 
            \n\n {recipient.chat_messages_for_summary(sender)[-1]['content']}'''

review_chats = [
    {
     "recipient": SEO_reviewer, 
     "message": reflection_message, 
     "summary_method": "reflection_with_llm",
     "summary_args": {"summary_prompt" : 
        "Return review into as JSON object only:"
        "{'Reviewer': '', 'Review': ''}. Here Reviewer should be your role",},
     "max_turns": 1},
    {
    "recipient": legal_reviewer, "message": reflection_message, 
     "summary_method": "reflection_with_llm",
     "summary_args": {"summary_prompt" : 
        "Return review into as JSON object only:"
        "{'Reviewer': '', 'Review': ''}.",},
     "max_turns": 1},
    {"recipient": ethics_reviewer, "message": reflection_message, 
     "summary_method": "reflection_with_llm",
     "summary_args": {"summary_prompt" : 
        "Return review into as JSON object only:"
        "{'reviewer': '', 'review': ''}",},
     "max_turns": 1},
     {"recipient": meta_reviewer, 
      "message": "Aggregrate feedback from all reviewers and give final suggestions on the writing.", 
     "max_turns": 1},
]

In [16]:
##Register Nested Chats to the Critic

In [17]:
critic.register_nested_chats(
    review_chats,
    trigger=writer,
)

In [18]:
##Full Orchestration with Nested Reflection

In [19]:
res = critic.initiate_chat(
    recipient=writer,
    message=task,
    max_turns=2,
    summary_method="last_msg"
)

Critic (to Writer):


        Write a concise but engaging blogpost about
      Hope AI . Make sure the blogpost is
       within 100 words.
       

--------------------------------------------------------------------------------
Writer (to Critic):

Title: Harnessing Tomorrow: The Hope of AI 

As we stand on the precipice of a new age, Artificial Intelligence (AI) inspires both excitement and trepidation. It's the great hope of our era - the potential to revolutionize everything from mundane tasks to major industries. Even as we grapple with questions of ethics and governance, the promise of AI remains. We envision a tomorrow where AI supports us, making life easier, unlocking human potential, and creating a world of possibilities. But, for all our aspirations, AI's real success lies in the cautious and responsible balances we strike today!


--------------------------------------------------------------------------------

*************************************************************